In [1]:
import csv
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.lines import Line2D
from PIL import Image

# process file

In [2]:
word_mapping = {
    'true': 1,
    'false': 0,
    'exec()': 0,
    'fork()': 1,
    'exit()': 2,
    'malloc()': 3,
    'free()': 4,
    'realloc()': 5,
    'ent:pthread_create()': 6,
    'ret:pthread_create()': 7,
    'ent:pthread_join()': 8,
    'ret:pthread_join()': 9,
    'ent:pthread_mutex_lock()': 10,
    'ret:pthread_mutex_lock()': 11,
    'ent:pthread_mutex_unlock()': 12,
    'ret:pthread_mutex_unlock()': 13
}

In [3]:
not_key_count = 0
val_err_count = 0
not_key_map = defaultdict(int)
err_words = defaultdict(int)
def debug_vals(vals):
    lo, hi = np.iinfo(np.int64).min, np.iinfo(np.int64).max
    for i, v in enumerate(vals):
        try:
            x = int(v)  # after your parse_key
        except Exception as e:
            print(f"[{i}] non-int: {v!r} ({type(v).__name__}) -> {e}")
            continue
        if not (lo <= x <= hi):
            print(f"[{i}] overflows int64: {v!r} = {x}")

def parse_key(key: str) -> int:
    global not_key_count, val_err_count, err_words
    if not key:
        not_key_map[key] += 1
        not_key_count += 1
        return -1
    if key in word_mapping:
        return word_mapping[key]
    # bug: lifetime is a float, not handled correctly yet
    try:
        return int(key)
    except ValueError:
        val_err_count += 1
        err_words[key] += 1
        return -1

def add_stats_line(arr: np.ndarray | None, line: list[str]) -> np.ndarray:
    if not line:
        return arr if arr is not None else np.empty((0, 0), dtype=int)

    vals = [parse_key(word) for word in line]
    debug_vals(vals)
    row = np.array(vals, dtype=np.int64)[None, :] # trick to add a dimension to row

    if arr is None or arr.size == 0:
        #print('arr is None')
        return row
    if arr.dtype != np.int64:
        arr = arr.astype(np.int64, copy=False)
    if arr.ndim == 1: # if dimension of arr is 1
        #print('arr is dimension 1')
        arr = arr[None, :]
    if arr.shape[1] != row.shape[1]:
        print(line)
        print(arr)
        print(row)
        raise ValueError(f'Column mismatch: arr has {arr.shape[1]} cols, row has {row.shape[1]} cols.')
    return np.vstack((arr,row))

In [25]:
stats = None
file = 'trace_1_CPU_multiproc_results.csv'
with open(file, 'r') as trace_csv:
    trace_reader = csv.reader(trace_csv)
    next(trace_reader)

    for line in trace_reader:
        if any('Lost' in s for s in line):
            continue
        stats = add_stats_line(stats, line)

print(f'{file}: {not_key_count=}, {val_err_count=}')
print(err_words)

trace_1_CPU_multiproc_results.csv: not_key_count=135848, val_err_count=2
defaultdict(<class 'int'>, {'Lost 18462 events': 2})


In [26]:
print(not_key_map.items())

dict_items([('', 135848)])


In [27]:
print(type(stats))
print(stats.shape)
print(stats.ndim)
print(stats[-5:])

<class 'numpy.ndarray'>
(0, 0)
2
[]


# plots

### mata configs

In [15]:
cpu_count = '_1_CPU'

### threads behaviors

In [16]:
threads_mem = {} # tid : ([time_ms], [memory_bytes])
program_mem = []
start_ms = stats[0][0] // 1000000
threads = set()

mem_used, with_realloc = 0, 0

for row in stats:
    tid = 0 if row[2] == 1 else row[1]
    # add threads
    if tid not in threads:
        threads.add(int(tid))
    if (row[3] == 0 or row[3] == 1) and tid not in threads_mem:
        threads_mem[tid] = ([],[]) #time_ms, memory_bytes
    if tid in threads_mem and row[3] in [3,4,5]: # malloc(), free(), realloc()
        if row[3] == 3:
            mem_used += row[6]
            with_realloc += row[6]
        if row[3] == 5:
            with_realloc += row[7] - row[6]
        timestamp = row[0] //1000000 - start_ms
        threads_mem[tid][0].append(timestamp)
        threads_mem[tid][1].append(row[5])
        program_mem.append((timestamp,row[4]))
print(f'{mem_used=}')
print(f'{with_realloc=}')
print(f'All threads recorded are: {threads}.')
print(f'Relevant threads are: {[int(tid) for tid in threads_mem.keys()]}.')

mem_used=np.int64(245578)
with_realloc=np.int64(245578)
All threads recorded are: {0, 17582, 17583, 17584, 17585, 17586}.
Relevant threads are: [0, 17582, 17583, 17584, 17585, 17586].


In [17]:
lines = {}
plt.figure(figsize=(10,5))
plt.xlabel('time since program starts (ms)')
plt.ylabel('memory allocated (bytes)')
plt.title('dynamic memory by time')

# all threads dynamic memory plot
for tid, (time, mem) in threads_mem.items():
    smooth_window = 3
    # np.convolve with mode=valid only returns indices that can be convolved, so the first n-1 items in list is dropped
    # therefore add a total of n-1=9 paddings to mem to balance out the cut
    pad_left, pad_right = smooth_window // 2 - 1 if smooth_window % 2 == 0 else smooth_window // 2, smooth_window // 2
    mem_padded = np.pad(mem, (pad_left, pad_right), mode='edge')
    # each number is now the average of nth to n-9th number in mem
    mem_smooth = np.convolve(mem_padded, np.ones(smooth_window)/smooth_window, mode='valid')
    style = '-' if tid != 0 else '--'
    line, = plt.plot(time, mem_smooth, linestyle=style, label=tid, alpha=0.8, lw=2) # interesting unpack returned list of lines into a single line with ,
    lines[tid] = line
plt.legend([lines[0]], ["main"])
#plt.yscale('symlog')
plt.savefig('plots/thread_memories_dashed' + cpu_count + '.png', dpi=300)
# create black and white version
# pil lib or other ways plt.show(cmap='gray')
plt.cla()
# convert to greyscale version
#with Image.open('plots/thread_memories_dashed.png') as im:
#    im.convert('L').save('plots/bw/threads_memories_dashed_bw.png')

# process dynamic memory
plt.title('process heap memory by time')
plt.xlabel('time since program starts (ms)')
plt.ylabel('memory allocated (bytes)')
time, mem = zip(*program_mem) #*iterable to unwrap
print(len(time), len(mem))
plt.plot(time, mem, label='process')
# draw vertical lines whenever a thread ends
for tid in threads_mem.keys():
    if tid == 0:
        continue
    plt.axvline(x=threads_mem[tid][0][-1], linestyle="--")
# create proxy lines for custom legend
legend_elements = [
    Line2D([0], [0], linestyle='--', label='thread ends'),
    Line2D([0], [0], linestyle='-', label='process heap')
]
plt.legend(handles=legend_elements, loc='upper left')
plt.savefig('plots/process_memory_threads_end' + cpu_count + '.png', dpi=300)
plt.clf()

plt.close()

66315 66315


### pthread lib

In [18]:
# [uprobe create time, uretprobe create time, create time used, uprobe join time, uretprobe join time, join time used]
# one for each thread
create_enter = []
create_end = []
create_durations = []
join_enter = []
join_end = []
join_durations = []
start_ns = stats[0][0]

for row in stats:
    tid = row[1]
    if row[3] == 6:
        create_enter.append(row[0] - start_ns)
    if row[3] == 7:
        create_end.append(row[0] - start_ns)
        create_durations.append(row[5])
    if row[3] == 8:
        join_enter.append(row[0] - start_ns)
    if row[3] == 9:
        join_end.append(row[0] - start_ns)
        print(f'join: {row[5]=}, {row[0]-start_ns=}')
        join_durations.append(row[5])

join: row[5]=np.int64(197299433), row[0]-start_ns=np.int64(1515265033)
join: row[5]=np.int64(15452), row[0]-start_ns=np.int64(1515287659)
join: row[5]=np.int64(514905447), row[0]-start_ns=np.int64(2030196853)


In [19]:
# box plot for time used in each function

fig, axes = plt.subplots(2, 1, sharex=False)

axes[0].boxplot(create_durations, vert=False)
axes[0].set_title('pthread_create()')

axes[1].boxplot(join_durations, vert=False)
axes[1].set_title('pthread_join()')

plt.xlabel("time (ns)")
plt.tight_layout()
plt.savefig('plots/pthread_functions_time' + cpu_count + '.png', dpi=300)
plt.close()

### mutex

In [20]:
# records (ent:lock, ret:unlock) timestamps of each mutex hold by threads
mutex_timestamp_map = defaultdict(list)
# records the duration (ns) from ent:mutex_lock() to ret:mutex_unlock()
mutex_duration_map = defaultdict(list)
# records the timestamp (ns) of most recent ent:mutex_lock to calculate the duration later
mutex_start_map = defaultdict(lambda: -1)
# records main thread tid too make it 0 in our data structures
main_tid = 0
# records the number of occurence where a ret:mutex_unlock() is recorded with no corresponding ent:mutex_lock()
missed_mutex_count = 0
missed_by_no_tid = 0
missed_by_neg = 0

for row in stats:
    tid = row[1] if row[2] == 0 else main_tid
    timestamp = row[0]
    if row[3] == 10: #ent:pthread_mutex_lock()
        mutex_start_map[tid] = timestamp
    elif row[3] == 13: #ret:pthread_mutex_unlock()
        # handle missing corresponding ent:mutex_lock()
        if tid not in mutex_start_map or mutex_start_map[tid] == -1:
            if tid not in mutex_start_map:
                missed_by_no_tid += 1
            if mutex_start_map[tid] == -1:
                missed_by_neg += 1
            missed_mutex_count += 1
            continue
        duration_ns = timestamp - mutex_start_map[tid]
        mutex_duration_map[tid].append(duration_ns)
        mutex_timestamp_map[tid].append( (timestamp, mutex_start_map[tid] ) )
        mutex_start_map[tid] = -1

print(f'the number of occurence where a ret:mutex_unlock() is recorded with no corresponding ent:mutex_lock() is {missed_mutex_count}')
print(f'{missed_by_no_tid=}, {missed_by_neg=}')

the number of occurence where a ret:mutex_unlock() is recorded with no corresponding ent:mutex_lock() is 9
missed_by_no_tid=0, missed_by_neg=9


##### duration histogram

In [21]:
mutex_duration_list = []
for l in mutex_duration_map.values():
    mutex_duration_list += l
print('number of mutex durations recorded is ' + str(len(mutex_duration_list)))

mutex_duration_min = min(mutex_duration_list)
mutex_duration_range = max(mutex_duration_list) - mutex_duration_min
bins_count = 8
bins_edges_list = [mutex_duration_min + (i * mutex_duration_range // bins_count) for i in range(bins_count + 1)]

plt.hist(mutex_duration_list, bins=bins_edges_list)

plt.xlabel('mutex hold duration (ns)')
plt.ylabel('occurence')
plt.title('durations of mutex hold over multithreads')

plt.savefig('plots/mutex_duration_hist' + cpu_count + '.png', dpi=300)
plt.close()

#with Image.open('plots/mutex_duration_hist.png') as img:
#    img.convert('L').save('plots/bw/mutex_duration_hist_bw.png')

number of mutex durations recorded is 2514


##### timeline plot

In [22]:
for tid in mutex_timestamp_map.keys():
    print([(int(i), int(j)) for i, j in mutex_timestamp_map[tid][:5]])

[(6717685201386, 6717685181196), (6717685414300, 6717685396402), (6717685547675, 6717685530328), (6717685744906, 6717685689857), (6717685887054, 6717685869950)]
[(6717692243718, 6717691963356), (6717692366513, 6717692272493), (6717692997782, 6717692385458), (6717693478747, 6717693018050), (6717697002960, 6717694069426)]
[(6717694116188, 6717693380650), (6717694472605, 6717694146969), (6717695095664, 6717694492796), (6717695523960, 6717695117320), (6717696203302, 6717695544437)]
[(6717699605612, 6717695849012), (6717699998355, 6717699627423), (6717703763111, 6717700620763), (6717704428887, 6717703783434), (6717704635970, 6717704472089)]
[(6717710480985, 6717702649543), (6717710788425, 6717710501164), (6717711134494, 6717710808219), (6717714515412, 6717712926081), (6717714694481, 6717714535658)]
[(6717761479933, 6717722638568), (6717761661062, 6717761500101), (6717761883718, 6717761680514), (6717762109978, 6717761902487), (6717762289642, 6717762128560)]


In [23]:
fig, ax = plt.subplots(figsize=(10, 5))

yticks = []
yticklabels = []
height = 0.8
ypos = 0

for tid, interval in mutex_timestamp_map.items():
    if tid == 0:
        continue
    ax.broken_barh(interval[:5], (ypos, height), facecolors="tab:blue")
    yticks.append(ypos + height / 2)
    yticklabels.append(tid)
    ypos += 1

ax.set_xlabel("Time (ns)")
ax.set_yticks(yticks)
ax.set_yticklabels(yticklabels)
ax.set_title("Mutex Hold Timeline")

plt.savefig('plots/mutex_timeline' + cpu_count + '.png')
plt.close()

##### word read latency by time

In [24]:
# timestamp is calculated by start_timestamp + (end_timestamp - start_timestamp) / 2
tid_latency_map = defaultdict(lambda: ([], [])) # [list for timestamp (ns)], [list for latency (ns)]

for tid in mutex_timestamp_map.keys():
    if tid == 0:
        continue
    for i in range(1, len(mutex_timestamp_map[tid])):
        latency_ms = (mutex_timestamp_map[tid][i][0] - mutex_timestamp_map[tid][i-1][0]) / 1000000
        timestamp_ms = mutex_timestamp_map[tid][i-1][0] / 1000000 + latency_ms / 2
        tid_latency_map[tid][0].append(timestamp_ms)
        tid_latency_map[tid][1].append(latency_ms)
for tid in tid_latency_map.keys():
    print(np.mean(tid_latency_map[tid]))

fig, ax = plt.subplots(figsize=(10,5))

#print(len(tid_latency_map))

for tid, data in tid_latency_map.items():
    #print(len(data[0]), len(data[1]))
    time_list, latency_list = data
    smooth_window = 1
    pad_left, pad_right = smooth_window // 2 - 1 if smooth_window % 2 == 0 else smooth_window // 2, smooth_window // 2
    latency_list_padded = np.pad(latency_list, (pad_left, pad_right), mode='edge')
    latency_list_smooth = np.convolve(latency_list_padded, np.ones(smooth_window)/smooth_window, mode='valid')
    ax.plot(time_list, latency_list_smooth, label=tid, alpha=0.8, lw=1)

ax.set_xlabel("Program time (ms)")
ax.set_ylabel("Acquire-to-acquire latency (ms)")
plt.legend()
plt.savefig('plots/mutex_latency_trend' + cpu_count + '.png')

plt.close()

3359498.860965542
3359232.510206967
3359273.0303494874
3359125.339727308
3358954.1300549866
